# <center>Big Data &ndash; Exercises &ndash; Solution</center>
## <center>Fall 2020 &ndash; Week 9 &ndash; ETH Zurich</center>
## <center>Spark Dataframes and SparkSQL</center>

## Introduction
For this exercise please create a Spark cluster on Azure as in the previous exercises. 
- Make sure to choose Spark version 2.3. 
- You can keep the default cluster configuration.
- If you have performance problems, check the yarn UI (```https://<cluster_name>.azurehdinsight.net/yarnui/hn/cluster```) and make sure that there are no unwanted applications hogging the resources.
- Pay attention to the cores and memory usage. Part of this exercise is to also get familiar with the yarnUI and cluster management.

## Getting the data


- Log in using ssh to your cluster:  ```ssh <ssh_user_name>@<cluster_name>-ssh.azurehdinsight.net```
- Download the data: ```wget https://bigdata2020exassets.blob.core.windows.net/ex09/orders.jsonl```
- Upload the data to HDFS: ```hdfs dfs -put orders.jsonl /tmp/```


After you have uploaded the dataset into the Azure Blob, upload this notebook onto the Spark Jupyter server  (`https://<cluster-name>.azurehdinsight.net/jupyter`).

## Spark Session
When you execute the first cell in this notebook, a spark session will be created automatically, you can print information about the session with the ```%%info``` magics

In [ ]:
print("Hello")

In [ ]:
%%info

By default the session is created with 3 Spark executors. We can change the configuration with the ```%%configure``` magics. Make sure you have enough vCores/Memory. You can see your total amount of available resources in yarnUI. 

In [ ]:
%%configure -f
{"executorMemory": "2G", "numExecutors": 6, "driverMemory": "4G"}

## <center>1. Spark Dataframes</center>

Spark Dataframes allow the user to perform simple and efficient operations on data, as long as the data is structured and has a schema. Dataframes are similar to relational tables in relational databases: conceptually a dataframe is a specialization of a Spark RDD with schema information attached. You can find more information in Karau, H. et al. (2015). Learning Spark, Chapter 9 (optional reading).

### 1.1. Data preprocessing

In [ ]:
import json

path = "/tmp/orders.jsonl"
orders_df = spark.read.json(path).cache()

The type of our dataset object is DataFrame

In [ ]:
type(orders_df)

Print the schema

In [ ]:
orders_df.printSchema()

Print one row

In [ ]:
orders_df.limit(1).collect()

You can access the underlying RDD object and use any functions you learned for Spark RDDs.

In [ ]:
orders_df.rdd.filter(lambda ordr: ordr.customer.last_name == "Landry").count()

### 1.2. Dataframe Operations
We perform some queries using operations on Dataframes ([Here](https://spark.apache.org/docs/2.3.0/sql-programming-guide.html#untyped-dataset-operations-aka-dataframe-operations) is a guide on DF Operations with a link to the [API Documentation](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html))

We can select columns and show the result

In [ ]:
orders_df.select("customer.first_name", "customer.last_name").limit(5).show()

As you can see we can navigate to the nested items with the dot

In [ ]:
orders_df.filter(orders_df["customer.last_name"] == "Landry").count()

How about nested arrays?

In [ ]:
orders_df.select("order_id", "items").orderBy("order_id").limit(5).show()

Let us try to find orders of a fan.

In [ ]:
orders_df.filter(orders_df["items.product"] == "fan").count()

The above code doesn't work! Use ```array contains``` instead.

In [ ]:
from pyspark.sql.functions import array_contains

orders_df.filter(array_contains("items.product", "fan")).count()

Let us try to unnest the data.

Unnest the products with explode.

Explode will generate as many rows as there are elements in the array and match them to other attributes.

In [ ]:
from pyspark.sql.functions import explode

orders_df.select(explode("items").alias("i"), "i.product", "order_id").orderBy("order_id").limit(5).show()

Now we can use this table to filter.

In [ ]:
exploded_df = orders_df.select(explode("items").alias("i"), "i.product", "order_id")
exploded_df.filter(exploded_df["product"] == "fan").count()

You might have tried to access the i.product column directly using a ```.filter``` right after the ```.select```. That, however, does not work, because the column is not available to ```orders_df``` when creating a clause like ```(orders_df["i.product"] == "fan")```. A possible workaround when using Dataframe operations is that of using a string clause in ```.filter```, so that the product column will be resolved after it has been added with the ```.select```.

In [ ]:
orders_df.select(explode("items").alias("i"), "i.product", "order_id").filter("product = 'fan'").count()

Project the nested columns

In [ ]:
orders_df.select(explode("items").alias("i"), "*").select(
    "order_id", "customer.*", "date", "i.*").limit(3).show()

### 1.3 Exercises

1) Find the average quantity at which each product is purchased. Only show the top 10 products by quantity. (Hint: you may need to import the function ```desc``` from ```pyspark.sql.functions``` to define descending order)

In [ ]:
from pyspark.sql.functions import desc

orders_df.select(explode("items").alias("i"), "*").select(
    "i.product", "i.quantity"
).groupBy("product").avg("quantity").orderBy(desc("avg(quantity)")).limit(10).show()

2) Find the most expensive order

In [ ]:
exploded_df = orders_df.select(explode("items").alias("i"), "*")
exploded_df.select(
    "order_id", (exploded_df["i.quantity"] * exploded_df["i.price"]).alias("total")
).groupBy("order_id").sum("total").orderBy(desc("sum(total)")).limit(1).show()

## <center>2. Spark SQL</center>

Spark SQL allows the users to formulate their queries using SQL. The requirement is the use of Dataframes, which as said before are similar to relational tables. In addition to a familiar interface, writing queries in SQL might provide better performance than RDDs, inheriting efficiency from the Dataframe operations, while also performing automatic optimization of queries.

In order to use sql we need to create a temporary table.

This table only exists for the current session.

In [ ]:
orders_df.registerTempTable("orders")

### 2.1 Queries

Finally, run SQL queries on the registered tables. We will run the same queries as during the previous section, but with SQL.

As you can see we can navigate to the nested items with the dot.

In [ ]:
%%sql
-- Finally, run SQL queries on the registered tables
-- As you can see we can navigate to the nested items with the dot
SELECT count(*)
FROM orders
WHERE orders.customer.last_name == "Landry"

How about nested arrays?

In [ ]:
%%sql
-- How about nested arrays?
SELECT order_id, items
FROM orders AS o
ORDER BY order_id
LIMIT 5

Let us try to find orders of a fan.

In [ ]:
%%sql 
SELECT count(*)
FROM orders
WHERE items.product = "fan"

The above code doesn't work! Use ```array contains``` instead.

In [ ]:
%%sql

SELECT count(*)
FROM orders
WHERE array_contains(items.product, "fan")

Let us try to unnest the data.

Unnest the products with explode.

Explode will generate as many rows as there are elements in the array and match them to other attributes.

In [ ]:
%%sql
SELECT explode(items) as i, i.product, order_id
FROM orders
ORDER BY order_id
limit 5

Now we can use this table to filter.

In [ ]:
%%sql
-- Filter on product
SELECT count(*)
    FROM (
    SELECT explode(items) as i, i.product, order_id
    FROM orders
    ORDER BY order_id
    )
WHERE product = "fan"

You might have tried to access the i.product column directly in the same ```SELECT``` clause. That, however, does not work, because the column is not available to the ```WHERE``` clause. In order to access the built columns directly, we need to unnest the data and make it part of our ```FROM``` clause. ```LATERAL VIEW``` lets us do just that, matching each non-array attribute to an unnested row from the array.  

In [ ]:
%%sql
SELECT *
FROM orders lateral view explode(items) as flat_items
WHERE flat_items.product = "fan"
ORDER BY order_id
LIMIT 3

Project the nested columns

In [ ]:
%%sql
SELECT order_id, customer.first_name, customer.last_name, date, flat_items.*
FROM orders lateral view explode(items) item_table as flat_items
WHERE flat_items.product = "fan"
ORDER BY order_id
LIMIT 3

Having built an unnested table, we can now easily aggregate over the previously nested columns

### 2.2 Exercises

1) Find the average quantity at which each product is purchased. Only show the top 10 products by quantity. 

In [ ]:
%%sql
SELECT flat_items.product, AVG(flat_items.quantity) as av_quantity
FROM orders lateral view explode(items) flat_table flat_items
GROUP BY flat_items.product
ORDER BY av_quantity DESC
LIMIT 10

2) Find the most expensive order

In [ ]:
%%sql
SELECT order_id, SUM(flat_items.quantity * flat_items.price) as total
FROM orders lateral view explode(items) flat_table flat_items
GROUP BY order_id
ORDER BY total desc
LIMIT 1

## <center>3. Exercise: PageRank (Optional)</center>

The PageRank algorithm, named after Google's Larry Page, assigns a measure of importance to each node (page) in a graph based on the importance of incoming edges (links). The importance of each edge is, in turn, derived from the importance of the source node and its out-degree. PageRank was designed to rank web pages based on hyperlinks between pages, but it can be also used to rank scientific articles, or influential users in a social network.

The algorithm maintains two datasets: one collection of (*pageID*, *linkList*) elements containing the list of neighbors of each page, and one collection of (*pageID*, *rank*) elements containing the current rank for each page. The algorithm proceeds as follows:
1. Initialize each page's rank to $1.0$.
2. On each iteration, have page $x$ send a contribution of $\frac{rank(x)}{numNeighbors(x)}$ to its neighbors (the pages it has links to).
3. Set each page's rank to $0.15 + 0.85 \times contributionsReceived$.

The algorithm runs multiple iterations (of step 2 and 3) until it converges.

Implement the PageRank algorithm in Spark for a simple dataset, running the loop for a fixed number of iterations.

For instance, you can use "parallelize" for that as follows: 
```
links = sc.parallelize([(1, 2),(1, 4),(2, 1),(2, 3),(3, 2)])
```
where 1,2,3,4 represents ids of pages.

### 3.1 Use Spark RDDs

In [ ]:
links = sc.parallelize([(1, 2),(1, 4),(2, 1),(2, 3),(3, 2)]).groupByKey().cache()

ranks = links.mapValues(lambda x: 1.0)

for i in range(10):
    contributions = links.join(ranks).flatMap(
        lambda (page_id, (links_l, rank)): [(dest, rank/len(list(links_l))) for dest in list(links_l)]
    )
    ranks = contributions.reduceByKey(lambda x, y: x + y).mapValues(lambda v: 0.15 + 0.85*v)
    
ranks.collect()

### 3.2 Use Spark DataFrames

In [ ]:
links = sc.parallelize([(1, 2),(1, 4),(2, 1),(2, 3),(3, 2)])
links_df = spark.createDataFrame(links).toDF("page_id", "linked_id").cache()

In [ ]:
links_c = links_df.groupBy("page_id").count()
ranks = links_df.selectExpr("page_id", "1.0 as rank")

for i in range(10):
    ranks = ranks.join(links_df, "page_id").join(links_c, "page_id").selectExpr(
        "*", "rank / count as contrib"
    ).groupBy("linked_id").sum("contrib").withColumnRenamed("sum(contrib)", "sum_contrib").selectExpr(
        "linked_id as page_id", "(0.15 + 0.85*sum_contrib) as rank"
    ).distinct()
    
ranks.show()

### 3.3 Use Spark SQL
Hint: you can use
```
new_df = spark.sql("... SQL query ...")
new_df.registerTempTable("new_table")
```
to perform a query inside a for loop and making the updated *new_table* available from SQL at every step

In [ ]:
links = sc.parallelize([(1, 2),(1, 4),(2, 1),(2, 3),(3, 2)])
links_df = spark.createDataFrame(links).toDF("page_id", "linked_id").cache()

links_df.registerTempTable("links")

In [ ]:
links_c = spark.sql("""
                SELECT page_id, COUNT(linked_id) AS count_links
                FROM links
                GROUP BY page_id
          """)
links_c.registerTempTable("links_c")

ranks = spark.sql("""
            SELECT page_id, 1.0 AS rank
            FROM links
            GROUP BY page_id
        """)
ranks.registerTempTable("ranks")

for i in range(10):
    ranks = spark.sql("""
                SELECT DISTINCT l.linked_id AS page_id, (0.15 + 0.85*SUM(r.rank/c.count_links)) AS rank
                FROM links AS l JOIN ranks AS r ON l.page_id=r.page_id
                    JOIN links_c AS c ON l.page_id=c.page_id
                GROUP BY l.linked_id
            """)
    ranks.registerTempTable("ranks")
    
ranks.show()